## Operaciones con datos y manejo de datos nulos

En esta sección aplicaremos diversas operaciones a un conjunto de datos. Este proceso, conocido como preparación de datos, es fundamental para poder realizar análisis posteriores, tales como visualizar los datos.

Usaremos el conjunto de datos de derrame de petróleo puedes encontrar información de este conjunto de datos en la siguiente dirección web:

[Descripción del conjunto de datos de derrame de petróleo](https://github.com/bayesianoues/intro_progra_biologia_2026/blob/main/Datasets/oil_spill_descripcion.md)

El archivo del conjunto de datos de derrame de petróleo esta en el repositorio y se puede acceder en la siguiente dirección web:

[Conjunto de datos de derrame de petróleo](https://raw.githubusercontent.com/bayesianoues/intro_progra_biologia_2026/refs/heads/main/Datasets/oil_spill.csv)

A continuación vamos a cargar el dataset de derrame de petróleo.


In [ ]:
import pandas as pd
pd.set_option('display.precision', 2)
import numpy as np

filename = 'https://raw.githubusercontent.com/bayesianoues/intro_progra_biologia_2026/refs/heads/main/Datasets/oil_spill.csv'

oil_spill = pd.read_csv(filename, header=None)

oil_spill.head()

In [ ]:
print(oil_spill.shape)

El conjunto de datos contiene 937 casos. Cada caso incluye 48 características numéricas derivadas de técnicas de visión por computadora, un identificador de la porción y una etiqueta de clase. El caso normal (sin derrame) se etiqueta como 0, mientras que la presencia de un derrame de petróleo se etiqueta como 1. El conjunto de datos está muy desequilibrado: 896 casos corresponden a "sin derrame" y solo 41 casos a "con derrame".

Las características varían significativamente en escala: algunas alcanzan valores de miles (por ejemplo, la segunda columna), mientras que otras son fraccionarias. Varias columnas contienen muy pocos valores únicos, lo que hace que este conjunto sea especialmente útil para practicar la preparación de datos.

Al revisar el archivo de datos oil_spill.csv, tenga en cuenta que la primera columna contiene números enteros que identifican la porción de imagen. Las 48 columnas restantes son características numéricas de valores reales, cada una con diferentes rangos y distribuciones.

## Identificar columnas que contienen un solo valor

Las columnas que tienen una sola observación o valor probablemente sean inútiles para el modelado de datos. Estas columnas o predictores se denominan predictores de varianza cero, ya que si midiéramos la varianza (valor promedio respecto a la media), esta sería cero.

Las columnas que tienen un solo valor para todas las filas no contienen información útil para el modelado. Dependiendo de la elección de preparación de datos y de los algoritmos de modelado, las variables con un solo valor también pueden causar errores o resultados inesperados. Puedes detectar las filas que tienen esta propiedad utilizando la función unique() de NumPy, que informará el número de valores únicos en cada columna.

In [ ]:
# Resumir el número de valores únicos en cada columna
for i in range(oil_spill.shape[1]):
  print(i, len(np.unique(oil_spill.iloc[:, i])))

Al ejecutar el anterior código, se carga el conjunto de datos directamente y se imprime el número de valores únicos para cada columna. Podemos ver que el índice de columna 22 solo tiene un valor y debería eliminarse lo cual podemos hacer con el siguiente código.

In [ ]:
# Recolectamos las columnas a borrar
cols_to_drop = []

for i in range(oil_spill.shape[1]):
    if len(np.unique(oil_spill.iloc[:, i])) == 1:
        cols_to_drop.append(i)

# Borrar las columnas que contienen un solo valor
oil_spill = oil_spill.drop(columns=cols_to_drop)

print("Columnas que se han borrado:", cols_to_drop)

## Considerar columnas que tienen muy pocos valores únicos

En el tema anterior, vimos que algunas columnas del conjunto de datos de ejemplo tenían muy pocos valores únicos. Por ejemplo, había columnas que solo tenían 2, 4 y 9 valores únicos. Esto podría tener sentido para variables ordinales o categóricas. Sin embargo, en este caso, el conjunto de datos solo contiene variables numéricas. Por lo tanto, que una columna tenga solo 2, 4 o 9 valores numéricos únicos podría ser sorprendente. Podemos referirnos a estas columnas o predictores como predictores de varianza cercana a cero, ya que su varianza no es cero, sino un número muy pequeño cercano a cero.

Dependiendo de la elección de preparación de datos y de los algoritmos de modelado, las variables con muy pocos valores numéricos también pueden causar errores o resultados inesperados. Para ayudar a identificar columnas de este tipo, puedes calcular el número de valores únicos de cada variable como un porcentaje del número total de filas en el conjunto de datos. Hagamos esto manualmente usando NumPy. El ejemplo completo se enumera a continuación.

In [ ]:
filename = 'https://raw.githubusercontent.com/bayesianoues/intro_progra_biologia_2026/refs/heads/main/Datasets/oil_spill.csv'

oil_spill = pd.read_csv(filename, header=None)


# número de filas
n_rows = len(oil_spill)

# iterar en las columnas
for col in oil_spill.columns:
    num_unique = oil_spill[col].nunique()
    percentage = (num_unique / n_rows) * 100
    print(f'{col}, {num_unique}, {percentage:.1f}%')

Al ejecutar el ejemplo, se informa el índice de la columna y el número de valores únicos para cada columna, seguido del porcentaje de valores únicos con respecto al total de filas en el conjunto de datos. Aquí podemos ver que algunas columnas tienen un porcentaje muy bajo de valores únicos, como por debajo del 1 por ciento.

Podemos actualizar el ejemplo para resumir únicamente aquellas variables que tengan valores únicos que representen menos del 1 por ciento del número de filas.

In [ ]:

n_rows = len(oil_spill)

for col in oil_spill.columns:
    num_unique = oil_spill[col].nunique()
    percentage = (num_unique / n_rows) * 100

    if percentage < 1:
        print(f'{col}, {num_unique}, {percentage:.1f}%')

Al ejecutar el ejemplo, podemos ver que 11 de las 50 variables tienen valores numéricos cuyos valores únicos representan menos del 1 por ciento del número de filas. Esto no significa que estas filas y columnas deban eliminarse, pero requieren atención adicional. Por ejemplo:

* ¿Quizás los valores únicos se pueden codificar como valores ordinales?

* ¿Quizás los valores únicos se pueden codificar como valores categóricos?

* ¿Quizás comparar el rendimiento del modelo con cada variable eliminada del conjunto de datos?

Por ejemplo, si quisiéramos eliminar todas las 11 columnas con valores únicos que representan menos del 1 por ciento de las filas, el siguiente ejemplo demuestra cómo hacerlo.

In [ ]:
print(oil_spill.shape)

# calcular el porcentaje de valores únicos por columna
percentages = oil_spill.nunique() / len(oil_spill) * 100

# seleccionar columnas para eliminar (< 1%)
to_drop = percentages[percentages < 1].index

print(list(to_drop))

# eliminar columnas
oil_spill = oil_spill.drop(columns=to_drop)

print(oil_spill.shape)

Al ejecutar el ejemplo, primero se carga el conjunto de datos y se informa el número de filas y columnas.
Se calcula el número de valores únicos para cada columna y se identifican aquellas columnas que tienen
un número de valores únicos inferior al 1 por ciento de las filas. En este caso, 11 columnas.
Luego, las columnas identificadas se eliminan del DataFrame y se informa el número de filas y columnas
en el DataFrame para confirmar el cambio.

## Manejo de datos nulos

Los datos del mundo real a menudo contienen valores faltantes. Los datos pueden tener valores faltantes por diversas razones, como observaciones que no fueron registradas o corrupción de los datos. Manejar los datos faltantes es importante porque muchas técnicas estadísticas no admiten datos con valores faltantes. En este tutorial, descubrirás cómo manejar datos faltantes para analizar datos. Específicamente, después de completar este cuaderno sabrás:

* Cómo marcar valores inválidos o corruptos como faltantes en tu conjunto de datos.

* Cómo eliminar filas con datos faltantes de tu conjunto de datos.


Usaremos el conjunto de datos de diabetes puedes encontrar información de este conjunto de datos en la siguiente dirección web:

[Descripción del conjunto de datos de diabetes](https://github.com/bayesianoues/intro_progra_biologia_2026/blob/main/Datasets/pima_indians_diabetes_descripcion.md)

El archivo del conjunto de datos diabetes y se puede acceder en la siguiente dirección web:

[Conjunto de datos de diabetes](https://raw.githubusercontent.com/bayesianoues/intro_progra_biologia_2026/refs/heads/main/Datasets/pima_indians_diabetes.csv)

A continuación vamos a cargar el conjunto de datos de diabetes.


In [ ]:
filename = 'https://raw.githubusercontent.com/bayesianoues/intro_progra_biologia_2026/refs/heads/main/Datasets/pima_indians_diabetes.csv'

diabetes = pd.read_csv(filename, header=None)

diabetes.head()

In [ ]:
print(diabetes.shape)

El conjunto de datos tiene 768 casos y 9 variables.

## Marcando valores ausentes

Con el siguiente código vamos a mostrar estadisticos descriptivos.

In [ ]:
diabetes.describe()

Podemos ver que hay columnas que tienen un valor mínimo de cero (0). En algunas columnas, un valor de cero no tiene sentido e indica un valor inválido o faltante.

Los valores faltantes se indican frecuentemente con entradas fuera de rango; tal vez un número negativo (por ejemplo, -1) en un campo numérico que normalmente es solo positivo, o un 0 en un campo numérico que normalmente nunca puede ser 0.

Específicamente, las siguientes columnas tienen un valor mínimo inválido de cero:

1. Concentración de glucosa plasmática

2. Presión arterial diastólica

3. Grosor del pliegue cutáneo del tríceps

4. Insulina sérica a las 2 horas

5. Índice de masa corporal

Confirmemos esto observando los datos sin procesar; el ejemplo imprime las primeras 20 filas de datos.

In [ ]:
diabetes.head(20)

Al ejecutar el ejemplo, podemos ver claramente valores de 0 en las columnas 2, 3, 4 y 5.

Podemos obtener un recuento del número de valores faltantes en cada una de estas columnas. Podemos hacerlo marcando como True todos los valores en el subconjunto del DataFrame que nos interesa y que tengan valor cero. Luego podemos contar el número de valores True en cada columna.

In [ ]:
# contar el número de valores faltantes para cada columna
num_missing = (diabetes[[1,2,3,4,5]] == 0).sum()
# reportar los resultados
print(num_missing)

Podemos ver que las columnas 1, 2 y 5 tienen solo unos pocos valores cero, mientras que las columnas 3 y 4 muestran muchos más, casi la mitad de las filas. Esto resalta que pueden ser necesarias diferentes estrategias para valores faltantes según la columna, por ejemplo, para asegurar que aún quede un número suficiente de registros para por ejemplo entrenar un modelo predictivo.

En Python, específicamente en Pandas, NumPy y Scikit-Learn, marcamos los valores faltantes como NaN. Los valores con valor NaN son ignorados en operaciones como suma, conteo, etc. Podemos marcar valores como NaN fácilmente con el DataFrame de Pandas utilizando la función replace() en un subconjunto de las columnas que nos interesan. Después de haber marcado los valores faltantes, podemos usar la función isnull() para marcar todos los valores NaN en el conjunto de datos como True y obtener un conteo de los valores faltantes por cada columna.

In [ ]:
# reemplazar los valores '0' con 'nan'
diabetes[[1,2,3,4,5]] = diabetes[[1,2,3,4,5]].replace(0, np.nan)
# contar el número de valores nan en cada columna
print(diabetes.isnull().sum())

## Eliminar filas con valores faltantes

La estrategia más simple para manejar datos faltantes es eliminar los registros que contienen un valor faltante.

Podemos hacer esto creando un nuevo DataFrame de Pandas con las filas que contienen valores faltantes eliminadas. Pandas proporciona la función dropna() que se puede usar para eliminar columnas o filas con datos faltantes. Podemos usar dropna() para eliminar todas las filas con datos faltantes, de la siguiente manera:


In [ ]:
# resumir la forma de los datos antes de eliminar las
# filas con valores faltantes
print(diabetes.shape)
# eliminar filas con valores faltantes
diabetes.dropna(inplace=True)
# resumir la forma de los datos con las filas faltantes eliminadas
print(diabetes.shape)

 Al ejecutar este ejemplo, podemos ver que el número de filas se ha reducido drásticamente de 768 en el conjunto de datos original a 392 con todas las filas que contienen un NaN eliminadas. Hay que tomar en cuenta que este método se muestra a efectos ilustrativas y en este caso representa una perdida significativa de información, ante esta situación se sugiere explorar métodos de imputación de valores ausentes los cuales estan fuera del alcance del curso.